# 02 - Feature Engineering

Technische Indikatoren & Feature-Erstellung für das ML-Modell.

**Ziele:**
- Technische Indikatoren berechnen (RSI, MACD, Bollinger Bands, etc.)
- Feature-Korrelationen analysieren
- Feature-Importance untersuchen
- Preprocessing-Pipeline testen

In [1]:
import structlog
structlog.configure(
    wrapper_class=structlog.make_filtering_bound_logger(0),
)

from backend.workflows.fetch_data import fetch_and_store

features_df, manifest = fetch_and_store(save=True)

print(f"Shape: {features_df.shape}")
print(f"Manifest Hash: {manifest.column_hash}")
print(f"Columns: {len(manifest.columns)}")
features_df.head()

2026-03-08 19:17:33 [info     ] downloading raw data           end=2026-03-08 start=2000-08-01 symbol=^GSPC


[*********************100%***********************]  1 of 1 completed

2026-03-08 19:17:35 [info     ] raw data downloaded            rows=6437
2026-03-08 19:17:35 [info     ] saved                          path=backend/data/raw/sp500_ohlcv.parquet rows=6437
2026-03-08 19:17:35 [info     ] computing target              
2026-03-08 19:17:35 [info     ] computing indicators          


2026-03-08 19:17:36 [info     ] calc_indicators: 33 rows dropped (NaN), 6404 remaining
2026-03-08 19:17:36 [info     ] fetching extra market data    
2026-03-08 19:17:49 [info     ] fetch_extra_market_data: filled 513 NaN values, 0 remaining
2026-03-08 19:17:49 [info     ] features ready                 columns=53 hash=daca9010a42ea73c rows=6404
2026-03-08 19:17:49 [info     ] saved                          path=backend/data/features/sp500_features.parquet rows=6404
Shape: (6404, 53)
Manifest Hash: daca9010a42ea73c
Columns: 53


Price,Close,Volume,Target,SMA 10,EMA 10,EMA 20,WMA 10,Momentum 10,SAR,RSI,...,^IBEX Close,SI=F Close,HG=F Close,NG=F Close,^TNX Close,^IRX Close,^FVX Close,^TYX Close,SPY Close,EFA Close
Date,,,,,,,,,,,,,,,,,,,,,
2000-09-18,1444.510010,9.625000e+08,0,1484.369006,1479.509808,1485.168848,1476.255833,-76.260010,1505.653274,31.968074,...,11188.087891,4.876,0.9185,5.295,5.871,5.95,5.900,5.956,91.801643,22.202042
2000-09-19,1459.900024,1.024900e+09,0,1479.651013,1475.944393,1482.762294,1471.806927,-47.179932,1496.730622,40.364915,...,11065.489258,4.883,0.9215,5.370,5.850,5.97,5.907,5.914,92.634598,22.202042
2000-09-20,1451.339966,1.104000e+09,0,1475.560010,1471.470861,1479.769691,1466.659464,-40.910034,1489.057141,37.586141,...,10901.289062,4.830,0.9015,5.320,5.892,5.97,5.953,5.962,91.950439,22.202042
2000-09-21,1449.050049,1.105400e+09,0,1470.214014,1467.394350,1476.844011,1461.839471,-53.459961,1479.759991,36.855211,...,10803.389648,4.872,0.9255,5.287,5.871,5.96,5.945,5.927,90.552292,22.202042
2000-09-22,1448.719971,1.185500e+09,0,1465.636011,1463.999008,1474.165531,1457.931463,-45.780029,1471.950384,36.744288,...,11029.389648,4.910,0.9195,5.131,5.829,5.97,5.905,5.904,92.198273,22.202042


## Datenqualität prüfen

In [2]:
import pandas as pd

nan_total = features_df.isna().sum().sum()
print(f"NaN gesamt: {nan_total}")
print(f"Duplikate: {features_df.index.duplicated().sum()}")
print(f"Zeitraum: {features_df.index.min()} → {features_df.index.max()}")
print(f"\nTarget-Verteilung:")
print(features_df["Target"].value_counts())
print(f"\nDtypes:")
print(features_df.dtypes.value_counts())

NaN gesamt: 0
Duplikate: 0
Zeitraum: 2000-09-18 00:00:00 → 2026-03-06 00:00:00

Target-Verteilung:
Target
1    3854
0    2550
Name: count, dtype: int64

Dtypes:
float64    52
int64       1
Name: count, dtype: int64


## Feature-Manifest validieren

In [3]:
errors = manifest.validate_against(features_df)
if errors:
    print("Manifest-Fehler:")
    for e in errors:
        print(f"  - {e}")
else:
    print(f"Manifest OK — {len(manifest.columns)} Spalten, Hash: {manifest.column_hash}")

print(f"\nSpalten:\n{manifest.columns}")

Manifest OK — 53 Spalten, Hash: daca9010a42ea73c

Spalten:
['%D', '%K', '%R', '+DMI', '-DMI', 'ADOSC', 'ADX', 'CCI', 'CL=F Close', 'CNY=X Close', 'Close', 'EFA Close', 'EMA 10', 'EMA 20', 'EURUSD=X Close', 'GBPUSD=X Close', 'GC=F Close', 'HG=F Close', 'JPY=X Close', 'MACD', 'MACD_HIST', 'MACD_SIGNAL', 'Momentum 10', 'NG=F Close', 'OBV', 'ROC', 'RSI', 'SAR', 'SI=F Close', 'SMA 10', 'SPY Close', 'Target', 'Volume', 'WMA 10', '^AXJO Close', '^BSESN Close', '^DJI Close', '^FCHI Close', '^FTSE Close', '^FVX Close', '^GDAXI Close', '^HSI Close', '^IBEX Close', '^IRX Close', '^IXIC Close', '^MXX Close', '^N225 Close', '^RUT Close', '^TNX Close', '^TYX Close', 'low_band', 'mid_band', 'up_band']


## DataStore Roundtrip testen

In [11]:
from backend.infra.database import get_data_store
from backend.core.features.manifest import FeatureManifest

store = get_data_store()

loaded = store.load_features()
print(f"Saved: {features_df.shape} → Loaded: {loaded.shape}")

loaded_manifest = FeatureManifest.from_dataframe(loaded)
assert loaded_manifest.column_hash == manifest.column_hash, "Column hash mismatch!"
print(f"Roundtrip OK — Hash matches: {loaded_manifest.column_hash}")
loaded

Saved: (6404, 53) → Loaded: (6404, 53)
Roundtrip OK — Hash matches: daca9010a42ea73c


,Close,Volume,Target,SMA 10,EMA 10,EMA 20,WMA 10,Momentum 10,SAR,RSI,...,^IBEX Close,SI=F Close,HG=F Close,NG=F Close,^TNX Close,^IRX Close,^FVX Close,^TYX Close,SPY Close,EFA Close
Date,,,,,,,,,,,,,,,,,,,,,
2000-09-18,1444.510010,9.625000e+08,0,1484.369006,1479.509808,1485.168848,1476.255833,-76.260010,1505.653274,31.968074,...,11188.087891,4.876000,0.9185,5.295,5.871,5.950,5.900,5.956,91.801643,22.202042
2000-09-19,1459.900024,1.024900e+09,0,1479.651013,1475.944393,1482.762294,1471.806927,-47.179932,1496.730622,40.364915,...,11065.489258,4.883000,0.9215,5.370,5.850,5.970,5.907,5.914,92.634598,22.202042
2000-09-20,1451.339966,1.104000e+09,0,1475.560010,1471.470861,1479.769691,1466.659464,-40.910034,1489.057141,37.586141,...,10901.289062,4.830000,0.9015,5.320,5.892,5.970,5.953,5.962,91.950439,22.202042
2000-09-21,1449.050049,1.105400e+09,0,1470.214014,1467.394350,1476.844011,1461.839471,-53.459961,1479.759991,36.855211,...,10803.389648,4.872000,0.9255,5.287,5.871,5.960,5.945,5.927,90.552292,22.202042
2000-09-22,1448.719971,1.185500e+09,0,1465.636011,1463.999008,1474.165531,1457.931463,-45.780029,1471.950384,36.744288,...,11029.389648,4.910000,0.9195,5.131,5.829,5.970,5.905,5.904,92.198273,22.202042
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-02,6881.620117,6.079080e+09,0,6883.923975,6890.293740,6894.350574,6890.520490,45.450195,6785.909590,48.647591,...,17878.900391,88.283997,5.8945,2.960,4.048,3.590,3.621,4.698,686.380005,103.300003
2026-03-03,6816.629883,6.442080e+09,0,6881.264941,6876.900312,6886.948604,6878.285201,-26.590332,6952.509766,43.035212,...,17062.400391,82.922997,5.7735,3.054,4.056,3.595,3.631,4.702,680.330017,100.089996
2026-03-04,6869.500000,5.252170e+09,0,6880.083936,6875.554800,6885.286832,6876.146120,-11.810059,6947.667969,48.264266,...,17487.000000,82.633003,5.8550,2.917,4.080,3.595,3.668,4.716,685.130005,101.379997


## Feature-Korrelation mit Target

In [10]:
corr = features_df.corr()["Target"].drop("Target").sort_values(ascending=False)

print(corr)


Price
^MXX Close        0.089159
OBV               0.083133
GC=F Close        0.069897
HG=F Close        0.067845
^BSESN Close      0.063725
EFA Close         0.063412
^HSI Close        0.062147
Volume            0.057975
^RUT Close        0.057801
^AXJO Close       0.056949
%R                0.054056
+DMI              0.053218
^GDAXI Close      0.053081
RSI               0.052048
^DJI Close        0.051597
MACD_HIST         0.048391
SPY Close         0.048171
^IXIC Close       0.047045
CCI               0.044409
Close             0.044360
WMA 10            0.043871
EMA 10            0.043690
up_band           0.043686
SMA 10            0.043522
EMA 20            0.043379
mid_band          0.042906
SI=F Close        0.042487
low_band          0.042030
SAR               0.041905
ROC               0.040610
Momentum 10       0.036185
ADX               0.034465
^N225 Close       0.034160
^FTSE Close       0.032936
ADOSC             0.027090
MACD              0.023318
%D                0.01

## Statistik der Indikatoren

In [6]:
indicator_cols = [
    "RSI", "ROC", "%R", "MACD", "CCI", "ADX", "+DMI", "-DMI",
    "%K", "%D", "OBV", "ADOSC",
]
existing = [c for c in indicator_cols if c in features_df.columns]
features_df[existing].describe().round(2)

Price,RSI,ROC,%R,MACD,CCI,ADX,+DMI,-DMI,%K,%D,OBV,ADOSC
count,6404.00,6404.00,6404.00,6404.00,6404.00,6404.00,6404.00,6404.00,6404.00,6404.00,6.404000e+03,6404.00
mean,54.08,0.29,-38.29,5.91,20.73,22.98,24.39,23.71,55.25,55.25,6.346838e+11,0.10
std,11.33,3.31,31.46,28.72,106.50,7.98,7.25,7.36,32.87,18.70,5.056301e+11,0.21
min,13.64,-25.88,-100.00,-237.02,-355.74,8.14,5.34,5.28,0.00,0.00,-6.862037e+10,-0.61
25%,45.87,-1.23,-65.49,-5.66,-63.64,16.98,18.92,18.41,25.11,42.01,1.537446e+11,-0.05
50%,55.27,0.64,-30.57,6.92,44.54,21.72,24.52,22.89,58.94,55.78,6.840153e+11,0.10
75%,62.63,2.11,-9.38,16.89,104.64,27.71,29.59,28.28,86.11,68.72,1.058484e+12,0.25
max,86.69,21.64,-0.00,121.08,318.65,55.05,51.35,62.28,100.00,99.29,1.623346e+12,0.75
